<a href="https://colab.research.google.com/github/samuelaojih/Google-Colab/blob/main/LANDSAT_7_NDVI_PROCESSING_WORKFLOW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================
# LANDSAT 7 NDVI PROCESSING WORKFLOW
# Cloud Masking + SLC-Off Gap Filling + Smoothing
# Vertical Flip Fix + Properly Georeferenced GeoTIFF Export
# ==============================================================
#
# This script performs the following:
# 1. Connects to Google Earth Engine
# 2. Loads a shapefile asset as ROI
# 3. Loads Landsat 7 Surface Reflectance data
# 4. Applies cloud masking and SLC-off masking
# 5. Calculates NDVI
# 6. Converts the collection to xarray
# 7. Fills scan line gaps using 2D interpolation
# 8. Applies temporal and spatial smoothing
# 9. Fixes vertical inversion issue for GIS compatibility
# 10. Exports properly georeferenced GeoTIFF files
# 11. Zips outputs for download
#
# ==============================================================
# STEP 1: INSTALL REQUIRED PACKAGES (Run in Colab if needed)
# ==============================================================

!pip install --upgrade xee
!pip install -U geemap
!pip install scipy rioxarray rasterio

# ==============================================================
# STEP 2: IMPORT LIBRARIES
# ==============================================================

import ee
import geemap
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate
from scipy.ndimage import uniform_filter, gaussian_filter
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
import zipfile
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ==============================================================
# STEP 3: INITIALIZE GOOGLE EARTH ENGINE
# ==============================================================

# Authenticate once (will prompt login)
ee.Authenticate()

# Initialize using your Earth Engine project ID
ee.Initialize(
    project='YOUR_PROJECT_ID',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

# ==============================================================
# STEP 4: LOAD SHAPEFILE ASSET (ROI)
# ==============================================================

# Replace with your uploaded asset path
shapefile_asset = "projects/YOUR_PROJECT/assets/YOUR_SHAPEFILE"

roi = ee.FeatureCollection(shapefile_asset).geometry()

print("ROI loaded successfully")

# ==============================================================
# STEP 5: FUNCTION TO CALCULATE NDVI WITH CLOUD & SLC MASKING
# ==============================================================

def add_ndvi_with_mask(img):
    """
    Calculates NDVI and removes:
    - Clouds
    - Cloud shadows
    - SLC-off gaps (Landsat 7 issue)
    """

    # Extract QA bands
    qa_pixel = img.select('QA_PIXEL')
    qa_radsat = img.select('QA_RADSAT')

    # Cloud bits
    dilated_cloud = qa_pixel.bitwiseAnd(1 << 1).neq(0)
    cloud_shadow  = qa_pixel.bitwiseAnd(1 << 3).neq(0)
    cloud         = qa_pixel.bitwiseAnd(1 << 4).neq(0)

    # SLC-off gaps (radiometric saturation band)
    slc_gap = qa_radsat.eq(1)

    # Combine masks
    invalid_mask = dilated_cloud.Or(cloud_shadow).Or(cloud).Or(slc_gap)

    # Scale reflectance
    sr = img.select('SR_B.*').multiply(2.75e-05).add(-0.2)

    # NDVI = (NIR - RED) / (NIR + RED)
    ndvi = sr.normalizedDifference(['SR_B4', 'SR_B3']).rename('ndvi')

    # Apply mask
    return ndvi.updateMask(invalid_mask.Not()) \
              .copyProperties(img, ['system:time_start'])

# ==============================================================
# STEP 6: LOAD LANDSAT 7 COLLECTION
# ==============================================================

landsat7 = (
    ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
    .filterDate('2020-01-01', '2021-12-31')
    .filterBounds(roi)
    .filter(ee.Filter.lt('CLOUD_COVER', 45))
    .map(add_ndvi_with_mask)
)

print("Images found:", landsat7.size().getInfo())

# ==============================================================
# STEP 7: CONVERT TO XARRAY DATASET
# ==============================================================

first_image = landsat7.first()
projection = first_image.select('ndvi').projection().getInfo()
crs_string = projection['crs']

ds = xr.open_dataset(
    landsat7,
    engine='ee',
    crs=crs_string,
    scale=30,
    geometry=roi
)

ds = ds.sortby('time')

print("Dataset shape:", ds.ndvi.shape)

# ==============================================================
# STEP 8: FILL LANDSAT 7 SCAN LINE GAPS (2D INTERPOLATION)
# ==============================================================

def fill_scanlines_2d(da):
    """
    Fills NaN gaps using 2D interpolation.
    """

    data = da.values.copy()
    mask = np.isnan(data)

    if not mask.any():
        return da

    y_valid, x_valid = np.where(~mask)
    values_valid = data[~mask]

    y_grid, x_grid = np.mgrid[0:data.shape[0], 0:data.shape[1]]

    interp = interpolate.griddata(
        (y_valid, x_valid),
        values_valid,
        (y_grid, x_grid),
        method='linear',
        fill_value=np.nan
    )

    # Fill remaining NaN using nearest neighbor
    if np.isnan(interp).any():
        interp_nearest = interpolate.griddata(
            (y_valid, x_valid),
            values_valid,
            (y_grid, x_grid),
            method='nearest'
        )
        interp[np.isnan(interp)] = interp_nearest[np.isnan(interp)]

    filled_data = data.copy()
    filled_data[mask] = interp[mask]

    # Light smoothing
    filled_data = uniform_filter(filled_data, size=3)

    return xr.DataArray(filled_data, dims=da.dims, coords=da.coords)

# Apply to each time step
filled_images = []
for i in range(len(ds.time)):
    print(f"Processing image {i+1}/{len(ds.time)}")
    filled = fill_scanlines_2d(ds.ndvi.isel(time=i))
    filled_images.append(filled)

filled_ndvi = xr.concat(filled_images, dim='time')
filled_ndvi = filled_ndvi.assign_coords(time=ds.time)

# ==============================================================
# STEP 9: TEMPORAL & SPATIAL SMOOTHING
# ==============================================================

# Temporal smoothing (rolling mean)
ndvi_temporal = filled_ndvi.rolling(
    time=2,
    min_periods=1,
    center=True
).mean()

# Spatial smoothing (Gaussian filter)
def apply_gaussian_smoothing(da, sigma=0.5):
    smoothed = np.zeros_like(da.values)
    for i in range(len(da.time)):
        smoothed[i] = gaussian_filter(da.values[i], sigma=sigma)
    return xr.DataArray(smoothed, dims=da.dims, coords=da.coords)

ndvi_final = apply_gaussian_smoothing(ndvi_temporal, sigma=0.5)

# ==============================================================
# STEP 10: FIX VERTICAL INVERSION ISSUE
# ==============================================================

# Flip Y-axis to ensure GIS compatibility
ndvi_flipped = ndvi_final.isel(Y=slice(None, None, -1))
ndvi_flipped = ndvi_flipped.assign_coords(
    Y=ndvi_final.Y.values[::-1]
)

print("Orientation corrected")

# ==============================================================
# STEP 11: EXPORT FUNCTION (PROPERLY GEOREFERENCED)
# ==============================================================

def export_geotiff(da, filepath, crs_string):

    data = da.values
    data = np.where(np.isnan(data), -9999, data)

    x_coords = da.X.values
    y_coords = da.Y.values

    transform = from_bounds(
        x_coords.min(),
        y_coords.min(),
        x_coords.max(),
        y_coords.max(),
        data.shape[1],
        data.shape[0]
    )

    with rasterio.open(
        filepath,
        'w',
        driver='GTiff',
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype=data.dtype,
        crs=CRS.from_string(crs_string),
        transform=transform,
        nodata=-9999,
        compress='lzw'
    ) as dst:
        dst.write(data, 1)

# ==============================================================
# STEP 12: EXPORT ALL IMAGES
# ==============================================================

output_folder = Path("NDVI_Output")
output_folder.mkdir(exist_ok=True)

for i, date in enumerate(
    ndvi_flipped.time.dt.strftime('%Y%m%d').values
):
    filename = output_folder / f"NDVI_{date}.tif"
    export_geotiff(
        ndvi_flipped.isel(time=i),
        filename,
        crs_string
    )

print("GeoTIFF export completed")

# ==============================================================
# STEP 13: ZIP OUTPUT FILES
# ==============================================================

zip_filename = f"NDVI_Processed_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in output_folder.glob("*.tif"):
        zipf.write(file, file.name)

print("ZIP file created:", zip_filename)

# ==============================================================
# STEP 14: DOWNLOAD (COLAB ONLY)
# ==============================================================

try:
    from google.colab import files
    files.download(zip_filename)
except:
    print("Download manually from file browser")

# ==============================================================
# END OF SCRIPT
# ==============================================================

print("All tasks completed successfully.")